In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp03/ex_8.py ──
"""
Shared infrastructure for MLFP03 Exercise 8 — Production ML + Drift +
Deployment.

Contains: data loading (Singapore credit scoring via MLFPDataLoader),
baseline LightGBM + isotonic calibration training, PSI/KS drift helpers,
and common output directory setup.

Technique-specific code (conformal quantile logic, dashboard rendering,
readiness checklist) stays in the per-technique files.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl  # noqa: F401  # re-exported for students
import lightgbm as lgb
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex8_production"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring (from MLFP02)
# ════════════════════════════════════════════════════════════════════════

RANDOM_SEED = 42


def load_credit_split() -> dict[str, Any]:
    """Load Singapore credit scoring data, preprocess, and return a split.

    Returns a dict with keys:
        X_train, y_train, X_test, y_test : numpy arrays
        feature_names                    : list[str]
        default_rate                     : float (train positive rate)
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    # Drop identifier columns before preprocessing — `customer_id` is a row
    # key, not a feature. Leaving it in the matrix causes drift noise to
    # dominate the top-variance feature list AND inflates AUC against the
    # model's pattern-recognition capability on IDs. Both are data-leakage
    # symptoms; the fix is to exclude the identifier at load time.
    id_columns = [c for c in ("customer_id", "application_id") if c in credit.columns]
    if id_columns:
        credit = credit.drop(id_columns)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target="default",
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(y_train.mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# BASELINE MODEL — LightGBM + isotonic calibration
# ════════════════════════════════════════════════════════════════════════
#
# Hyperparameters come from Exercise 7's Bayesian optimisation run. These
# are frozen here so every technique file trains an identical baseline
# model and can be compared apples-to-apples.


def build_baseline_model(y_train: np.ndarray) -> lgb.LGBMClassifier:
    """Return an unfit LightGBM classifier configured for credit default."""
    base_rate = float(y_train.mean())
    scale_pos_weight = (1.0 - base_rate) / max(base_rate, 1e-6)
    return lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_SEED,
        verbose=-1,
    )


def train_calibrated_model(
    X_train: np.ndarray, y_train: np.ndarray
) -> CalibratedClassifierCV:
    """Train LightGBM and wrap with isotonic calibration (cv=5)."""
    base = build_baseline_model(y_train)
    base.fit(X_train, y_train)
    calibrated = CalibratedClassifierCV(base, method="isotonic", cv=5)
    calibrated.fit(X_train, y_train)
    return calibrated


def evaluate_classification(
    y_true: np.ndarray, y_proba: np.ndarray, threshold: float = 0.5
) -> dict[str, float]:
    """Return the full classification metric bundle used across ex_8."""
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


# ════════════════════════════════════════════════════════════════════════
# DRIFT STATISTICS — PSI + KS
# ════════════════════════════════════════════════════════════════════════
#
# PSI (Population Stability Index):
#     PSI = Σ (p_new - p_ref) * ln(p_new / p_ref)
#     < 0.1 no shift, 0.1-0.2 moderate, > 0.2 significant
# KS (Kolmogorov-Smirnov): two-sample test on the empirical CDFs.


def compute_psi(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Compute Population Stability Index on 1-D arrays."""
    _, bin_edges = np.histogram(reference, bins=bins)
    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)
    # Laplace-smooth to avoid log(0)
    ref_props = (ref_counts + 1) / (len(reference) + bins)
    cur_props = (cur_counts + 1) / (len(current) + bins)
    return float(np.sum((cur_props - ref_props) * np.log(cur_props / ref_props)))


def compute_ks(reference: np.ndarray, current: np.ndarray) -> tuple[float, float]:
    """Return (KS statistic, p-value) on 1-D arrays."""
    ks_stat, p_value = stats.ks_2samp(reference, current)
    return float(ks_stat), float(p_value)


def drift_row(
    reference: np.ndarray, current: np.ndarray, psi_threshold: float = 0.1
) -> dict[str, float | str]:
    """Return {psi, ks_stat, ks_pval, drift} for a single feature."""
    psi = compute_psi(reference, current)
    ks_stat, ks_pval = compute_ks(reference, current)
    drift = "YES" if (psi > psi_threshold or ks_pval < 0.05) else "No"
    return {"psi": psi, "ks_stat": ks_stat, "ks_pval": ks_pval, "drift": drift}


def simulate_gradual_drift(
    X_ref: np.ndarray, X_new: np.ndarray, n_features: int = 3, shift: float = 0.5
) -> np.ndarray:
    """Return a copy of X_new with mean-shifted top features (drift sim)."""
    drifted = X_new.copy()
    for i in range(n_features):
        drifted[:, i] += shift * X_ref[:, i].std()
    return drifted


def simulate_sudden_drift(
    X_ref: np.ndarray,
    X_new: np.ndarray,
    feature_idx: int = 0,
    sigma_shift: float = 3.0,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Replace one column in X_new with a heavily shifted Gaussian."""
    rng = np.random.default_rng(seed)
    drifted = X_new.copy()
    drifted[:, feature_idx] = rng.normal(
        loc=X_ref[:, feature_idx].mean() + sigma_shift * X_ref[:, feature_idx].std(),
        scale=X_ref[:, feature_idx].std(),
        size=drifted.shape[0],
    )
    return drifted




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 8.1: Conformal Prediction for Credit Default
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Train a calibrated LightGBM credit-default model end-to-end
#   - Apply split conformal prediction for distribution-free uncertainty
#   - Prove the 1-α coverage guarantee on held-out data
#   - Sweep α and read the cost of tighter coverage in set size
#   - Translate "ambiguous prediction set" into a business routing rule
#
# PREREQUISITES: MLFP03 Exercises 1-7, MLFP02 (preprocessing).
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory     — why conformal works without distributional assumptions
#   2. Build      — train calibrated LightGBM + compute nonconformity scores
#   3. Train      — calibrate q̂, generate prediction sets, measure coverage
#   4. Visualise  — plot coverage + singleton rate across α values
#   5. Apply      — DBS Bank Singapore: which applicants go to human review
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go



## THEORY — Why conformal prediction works


In [ ]:
# Every classifier outputs a probability, but a probability is not a
# guarantee. Conformal prediction instead delivers a prediction SET C(x)
# with P(Y ∈ C(X)) ≥ 1 - α — guaranteed, distribution-free, from only
# the exchangeability assumption.
#
# ALGORITHM (split conformal, binary classification):
#   1. Split test into CALIBRATION and EVALUATION halves
#   2. Nonconformity score s_i = 1 - p(y_true | x_i) on calibration
#   3. q̂ = ceil((n+1)(1-α)) / n quantile of s_i
#   4. On new x: include class c iff 1 - p(c|x) ≤ q̂
#
# Singleton = confident auto-decide. Set of size 2 = ambiguous → human.



## TASK 2 — BUILD: train the calibrated baseline model


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 8.1 — Conformal Prediction")
print("=" * 70)

split = load_credit_split()
X_train, y_train = split["X_train"], split["y_train"]
X_test, y_test = split["X_test"], split["y_test"]
feature_names = split["feature_names"]

print(f"\nData: train={X_train.shape}, test={X_test.shape}")
print(f"Default rate: {split['default_rate']:.1%}")

# TODO: Train the calibrated model by calling train_calibrated_model(X_train, y_train)
calibrated_model = ____

# TODO: Score the test set — use predict_proba(X_test)[:, 1] for the positive-class probability
y_proba = ____

# TODO: Compute the metric bundle with evaluate_classification(y_test, y_proba)
metrics = ____

print("\n=== Calibrated Model Metrics ===")
for k, v in metrics.items():
    print(f"  {k:<10} {v:.4f}")



### Checkpoint 1


In [ ]:
assert metrics["auc_roc"] > 0.5, "Task 2: Should beat random"
assert (
    0 < metrics["brier"] < 0.25
), "Task 2: Brier should be reasonable for calibrated model"
print("\n[ok] Checkpoint 1 — calibrated LightGBM trained and evaluated\n")



## TASK 3 — TRAIN the conformal calibrator


In [ ]:
n_cal = X_test.shape[0] // 2
X_cal, X_eval = X_test[:n_cal], X_test[n_cal:]
y_cal, y_eval = y_test[:n_cal], y_test[n_cal:]

cal_proba = calibrated_model.predict_proba(X_cal)[:, 1]
# TODO: Nonconformity score = 1 - p(TRUE class | x). Use np.where(y_cal == 1, 1 - cal_proba, cal_proba)
cal_scores = ____

alpha = 0.10  # target 90% coverage
n_cal_size = len(cal_scores)

# TODO: Compute quantile level ceil((n+1)(1-α)) / n, then q_hat = quantile of cal_scores at that level
quantile_level = ____
q_hat = float(np.quantile(cal_scores, min(quantile_level, 1.0)))

print(f"=== Conformal Calibration ===")
print(f"  Calibration set:   {n_cal_size} samples")
print(f"  Target coverage:   {1 - alpha:.0%}")
print(f"  Calibration q̂:     {q_hat:.4f}")

# Build prediction sets on evaluation data
eval_proba = calibrated_model.predict_proba(X_eval)[:, 1]
prediction_sets: list[set[int]] = []
for i in range(len(y_eval)):
    pset: set[int] = set()
    # TODO: Include class 1 iff (1 - eval_proba[i]) <= q_hat
    # TODO: Include class 0 iff eval_proba[i] <= q_hat
    # TODO: If empty, fall back to argmax class (1 if eval_proba[i] >= 0.5 else 0)
    ____
    prediction_sets.append(pset)

# TODO: Coverage = fraction of y_eval[i] in prediction_sets[i]
coverage = ____
avg_set_size = float(np.mean([len(ps) for ps in prediction_sets]))
singleton_rate = float(np.mean([len(ps) == 1 for ps in prediction_sets]))

print(f"\n=== Empirical Results on Evaluation Half ===")
print(f"  Coverage:          {coverage:.4f} (target: {1 - alpha:.2f})")
print(f"  Avg set size:      {avg_set_size:.3f}")
print(f"  Singleton rate:    {singleton_rate:.1%} (auto-decide)")
print(f"  Ambiguous rate:    {1 - singleton_rate:.1%} (route to human)")



### Checkpoint 2


In [ ]:
assert coverage >= (
    1 - alpha - 0.05
), f"Task 3: Coverage {coverage:.4f} should be near target {1 - alpha:.4f}"
assert 0 < avg_set_size <= 2, "Task 3: Set size should be between 0 and 2"
print("\n[ok] Checkpoint 2 — conformal coverage guarantee verified\n")



## TASK 4 — VISUALISE coverage vs α


In [ ]:
print("=== Coverage vs α Sweep ===")
print(f"{'Alpha':>8} {'Target':>10} {'Actual':>10} {'Avg Size':>10} {'Singleton':>12}")
print("─" * 54)

alphas_sweep = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
sweep_rows = []
for a in alphas_sweep:
    q_level = np.ceil((n_cal_size + 1) * (1 - a)) / n_cal_size
    q = float(np.quantile(cal_scores, min(q_level, 1.0)))
    sets = []
    for i in range(len(y_eval)):
        ps: set[int] = set()
        if (1 - eval_proba[i]) <= q:
            ps.add(1)
        if eval_proba[i] <= q:
            ps.add(0)
        if not ps:
            ps.add(1 if eval_proba[i] >= 0.5 else 0)
        sets.append(ps)
    cov = float(np.mean([y_eval[i] in s for i, s in enumerate(sets)]))
    avg_sz = float(np.mean([len(s) for s in sets]))
    single = float(np.mean([len(s) == 1 for s in sets]))
    sweep_rows.append(
        {"alpha": a, "coverage": cov, "avg_size": avg_sz, "singleton": single}
    )
    print(f"{a:>8.2f} {1 - a:>10.2f} {cov:>10.4f} {avg_sz:>10.3f} {single:>11.1%}")

xs = [r["alpha"] for r in sweep_rows]
fig = go.Figure()
# TODO: Add a line trace for empirical coverage (x=xs, y=coverage values)
____
# TODO: Add a dashed reference trace for target 1-α (theoretical)
____
fig.add_trace(
    go.Bar(
        x=xs,
        y=[r["singleton"] for r in sweep_rows],
        name="Singleton rate (auto-decide)",
        marker_color="#10b981",
        opacity=0.55,
        yaxis="y2",
    )
)
fig.update_layout(
    title="Conformal Prediction: Coverage vs α (Singapore credit default)",
    xaxis_title="α (risk budget)",
    yaxis=dict(title="Coverage", range=[0, 1.05]),
    yaxis2=dict(title="Singleton rate", overlaying="y", side="right", range=[0, 1.05]),
    legend=dict(orientation="h", y=-0.2),
    height=500,
)
viz_path = OUTPUT_DIR / "ex8_01_conformal_sweep.html"
fig.write_html(str(viz_path))
print(f"\nSaved: {viz_path}")



### Checkpoint 3


In [ ]:
assert len(sweep_rows) == len(alphas_sweep), "Task 4: Should sweep all alphas"
print("\n[ok] Checkpoint 3 — α sweep visualised\n")



## TASK 5 — APPLY: DBS Bank Singapore auto-decision routing


In [ ]:
# SCENARIO: DBS Bank's consumer credit team processes ~18,000 unsecured
# loan applications/month. MAS Notice 635 requires automated decisions
# be "explainable and contestable" — high-risk applications need human
# review.
#
# Conformal prediction gives DBS a MATHEMATICAL routing rule:
#   Prediction set = {0}   → auto-approve (90% confident: no default)
#   Prediction set = {1}   → auto-reject  (90% confident: default)
#   Prediction set = {0,1} → HUMAN REVIEW (model refuses to commit)

n_auto = int(round(singleton_rate * len(y_eval)))
n_human = len(y_eval) - n_auto
print(f"=== Routing decision at α=0.10 on {len(y_eval)} held-out applicants ===")
print(f"  Auto-decide (singleton):  {n_auto:>4}  ({singleton_rate:.1%})")
print(f"  Route to human review:    {n_human:>4}  ({1 - singleton_rate:.1%})")

# DOLLAR IMPACT
monthly_apps = 18_000
analyst_hourly = 85.0
time_per_review_h = 0.25
ambiguous_monthly = monthly_apps * (1 - singleton_rate)
status_quo_cost = monthly_apps * time_per_review_h * analyst_hourly
conformal_cost = ambiguous_monthly * time_per_review_h * analyst_hourly
savings = status_quo_cost - conformal_cost
print(f"\n  Analyst cost (status quo, 100% review): S${status_quo_cost:>10,.0f}/mo")
print(f"  Analyst cost (conformal routing):       S${conformal_cost:>10,.0f}/mo")
print(f"  Monthly savings:                         S${savings:>10,.0f}/mo")
print(f"  Annualised:                              S${savings * 12:>10,.0f}/yr")



## REFLECTION


[x] Trained a calibrated LightGBM credit-default model
  [x] Built split conformal prediction from nonconformity scores
  [x] Verified {coverage:.0%} empirical coverage at α={alpha}
  [x] Swept α and visualised the cost of tighter coverage
  [x] Translated prediction sets into DBS routing with S${savings * 12:,.0f}/yr savings

  KEY INSIGHT: A probability is a guess. A prediction set is a contract.

  Next: 02_drift_monitoring.py — what happens when exchangeability breaks.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

